In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('../data/processed/interactions_filtered.csv')
print(df.shape)
df.head()

(40879, 10)


,user_id,book_id,review_id,is_read,rating,review_text_incomplete,date_added,date_updated,read_at,started_at
0,8842281e1d1347389f2ab93d60773d4d,23310161,f4b4b050f4be00e9283c92a814af2670,True,4,Fun sequel to the original.,Tue Nov 17 11:37:35 -0800 2015,Tue Nov 17 11:38:05 -0800 2015,NaN,NaN
1,8842281e1d1347389f2ab93d60773d4d,817720,75fd46041466ceb406b7fd69b089b9c5,True,5,NaN,Wed May 20 21:29:23 -0700 2015,Wed May 20 21:29:23 -0700 2015,NaN,NaN
2,8842281e1d1347389f2ab93d60773d4d,1969280,5809d5592ee32745e048a9c67ac27100,True,5,NaN,Sat Nov 08 08:56:58 -0800 2014,Wed Dec 17 00:37:25 -0800 2014,NaN,NaN
3,8842281e1d1347389f2ab93d60773d4d,17290220,22d424a2b0057b18fb6ecf017af7be92,True,5,One of my favorite books to read to my 5 year ...,Sat Nov 08 08:54:03 -0800 2014,Wed Jan 25 13:56:12 -0800 2017,Tue Jan 24 00:00:00 -0800 2017,NaN
4,8842281e1d1347389f2ab93d60773d4d,1027760,0c8a75acde799d70696f4aecf2d611de,True,4,NaN,Tue Mar 18 22:23:03 -0700 2014,Wed Mar 22 11:47:31 -0700 2017,NaN,NaN


In [2]:
# For each book, compute number of ratings and average rating
popularity = df.groupby('book_id').agg(
    num_ratings=('rating', 'count'),
    avg_rating=('rating', 'mean')
).reset_index()

# Weighted score: balances average rating with number of ratings
# A book with 100 ratings of 4.0 should rank higher than one with 2 ratings of 5.0
C = popularity['avg_rating'].mean()  # global mean rating
m = popularity['num_ratings'].quantile(0.75)  # minimum ratings threshold (75th percentile)

popularity['score'] = (
    (popularity['num_ratings'] / (popularity['num_ratings'] + m)) * popularity['avg_rating'] +
    (m / (popularity['num_ratings'] + m)) * C
)

popularity = popularity.sort_values('score', ascending=False).reset_index(drop=True)
print(f"Global mean rating: {C:.2f}")
print(f"Min ratings threshold (m): {m:.0f}")
popularity.head(10)

Global mean rating: 3.99
Min ratings threshold (m): 16


,book_id,num_ratings,avg_rating,score
0,5,1004,4.520916,4.512676
1,11387515,153,4.549020,4.496603
2,113946,202,4.490099,4.453739
3,44186,97,4.525773,4.450570
4,16101018,45,4.600000,4.441181
5,30119,479,4.455115,4.440226
6,105549,54,4.537037,4.412929
7,240007,13,4.923077,4.410496
8,7784,178,4.432584,4.396396
9,90072,85,4.470588,4.395055


In [3]:
def recommend_popular(n=10, min_ratings=None):
    """Return top-N most popular books."""
    results = popularity.copy()
    if min_ratings:
        results = results[results['num_ratings'] >= min_ratings]
    return results.head(n)[['book_id', 'num_ratings', 'avg_rating', 'score']]

recommend_popular(n=10)

,book_id,num_ratings,avg_rating,score
0,5,1004,4.520916,4.512676
1,11387515,153,4.549020,4.496603
2,113946,202,4.490099,4.453739
3,44186,97,4.525773,4.450570
4,16101018,45,4.600000,4.441181
5,30119,479,4.455115,4.440226
6,105549,54,4.537037,4.412929
7,240007,13,4.923077,4.410496
8,7784,178,4.432584,4.396396
9,90072,85,4.470588,4.395055


In [5]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

train, test = train_test_split(df, test_size=0.2, random_state=42)

# Baseline prediction: for any book, predict its average rating from train set
train_avg = train.groupby('book_id')['rating'].mean().to_dict()
global_mean = train['rating'].mean()

# For books in test not seen in train, fall back to global mean
test = test.copy()
test['predicted'] = test['book_id'].map(train_avg).fillna(global_mean)

rmse = np.sqrt(mean_squared_error(test['rating'], test['predicted']))
print(f"Baseline RMSE: {rmse:.4f}")
print(f"(This is the score your collaborative filter needs to beat)")

Baseline RMSE: 0.9060
(This is the score your collaborative filter needs to beat)


In [6]:
# Benchmark to beat
BASELINE_RMSE = 0.9060
print(f"Baseline RMSE: {BASELINE_RMSE}")
print(f"This is what a naive 'average rating per book' model scores.")
print(f"Any collaborative filter worth using should score below this.")

Baseline RMSE: 0.906
This is what a naive 'average rating per book' model scores.
Any collaborative filter worth using should score below this.
